[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/04_llm_finetuning/inference_optimization.ipynb)

# 07 . Inference Optimization (KV cache, batching, speculative decoding)

## Concept — Inference Optimization (revision notes)

**What it is**
- Techniques that make autoregressive generation cheaper: the **KV cache** (never recompute keys/values of past tokens), **batching** (serve many sequences per forward pass), and **speculative decoding** (a cheap draft model proposes several tokens, the big model verifies them in one pass).

**Why it matters**
- Generation is sequential: token `t+1` needs token `t`. Without a cache every step re-processes the whole prefix, making total cost O(n^2) in length. At inference the bottleneck is usually memory bandwidth (loading weights and cache for each token), not arithmetic, so anything that amortises those loads wins.

**When to use**
- KV cache: always, for any autoregressive decoding. Batching: whenever you serve more than one request (throughput). Speculative decoding: latency-sensitive single-stream decoding when you have a good small draft model.

**Math & intuition**
- Cache size per sequence: `2 (K and V) * layers * heads * head_dim * seq_len * bytes`. It grows linearly with context and is often the real limit on batch size.
- With cache, step `t` does O(t) attention work (one query vs. t keys) instead of recomputing O(t) tokens' worth of everything.
- Batching with different prompt lengths needs **padding + attention masks + correct position ids**; otherwise padded tokens leak into the result.
- Speculative decoding with `K` drafted tokens and per-token acceptance rate `alpha` yields on average `(1 - alpha^(K+1)) / (1 - alpha)` tokens per target-model pass. The accept/reject rule (accept with prob `min(1, p/q)`, else resample from `norm(max(0, p - q))`) makes the output distribution **exactly** the target's.

Everything below uses a small randomly initialised GPT defined in the notebook (no downloads); timings are CPU, single-thread, so exact numbers vary by machine.

### 1. A tiny decoder-only GPT whose forward pass supports a KV cache

**What this cell does (plain language):** we write attention by hand (not `nn.MultiheadAttention`) so the cache is visible: `forward` takes optional cached keys/values for each layer, appends the new tokens' keys/values, and returns the updated cache. It also accepts position ids and a key-validity mask, which we need for batching later. The model is randomly initialised (the head is scaled up so predictions are peaked and greedy decoding is stable), which is all we need to study *speed* and *equivalence*.

In [1]:
import torch
torch.set_num_threads(1)  # tiny models: multi-threading overhead only slows them down (and keeps runs deterministic)
import time, math, copy, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)

class Block(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.proj = nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.fc1, self.fc2 = nn.Linear(d, 4 * d), nn.Linear(4 * d, d)
        self.heads = heads
    def forward(self, h, past_kv, key_valid):
        B, T, d = h.shape; H = self.heads
        q, k, v = self.qkv(self.ln1(h)).chunk(3, dim=-1)
        sp = lambda t: t.view(B, T, H, d // H).transpose(1, 2)      # (B, H, T, hd)
        q, k, v = sp(q), sp(k), sp(v)
        past = 0
        if past_kv is not None:                                      # append new K/V to the cached ones
            past = past_kv[0].shape[2]
            k, v = torch.cat([past_kv[0], k], dim=2), torch.cat([past_kv[1], v], dim=2)
        S = k.shape[2]
        scores = q @ k.transpose(-1, -2) / math.sqrt(d // H)          # (B, H, T, S)
        qi = torch.arange(past, past + T)[:, None]; kj = torch.arange(S)[None, :]
        mask = (kj <= qi)[None, None]                                 # causal: query i sees keys <= its absolute position
        if key_valid is not None: mask = mask & key_valid[:, None, None, :]   # hide padding keys
        scores = scores.masked_fill(~mask, -1e9)                      # -1e9 (not -inf) so fully-masked pad rows stay finite
        a = (scores.softmax(-1) @ v).transpose(1, 2).reshape(B, T, d)
        h = h + self.proj(a)
        h = h + self.fc2(F.gelu(self.fc1(self.ln2(h))))
        return h, (k, v)

class GPT(nn.Module):
    def __init__(self, V=64, d=128, layers=4, heads=4, maxlen=1024):
        super().__init__()
        self.tok, self.pos = nn.Embedding(V, d), nn.Embedding(maxlen, d)
        self.blocks = nn.ModuleList([Block(d, heads) for _ in range(layers)])
        self.lnf, self.head = nn.LayerNorm(d), nn.Linear(d, V, bias=False)
        with torch.no_grad(): self.head.weight.mul_(8)               # peaked logits -> stable greedy decoding
    def forward(self, ids, cache=None, key_valid=None, pos=None):
        B, T = ids.shape
        past = 0 if cache is None else cache[0][0].shape[2]
        if pos is None: pos = torch.arange(past, past + T)[None].expand(B, T)
        h = self.tok(ids) + self.pos(pos)
        new_cache = []
        for i, blk in enumerate(self.blocks):
            h, kv = blk(h, None if cache is None else cache[i], key_valid)
            new_cache.append(kv)
        return self.head(self.lnf(h)), new_cache

model = GPT().eval()
print("parameters:", sum(p.numel() for p in model.parameters()))
ids = torch.randint(0, 64, (1, 10))
logits, cache = model(ids)
print("logits:", tuple(logits.shape), "| cache: layers", len(cache), "| K shape per layer", tuple(cache[0][0].shape), "(B, heads, seq, head_dim)")


parameters: 940800
logits: (1, 10, 64) | cache: layers 4 | K shape per layer (1, 4, 10, 32) (B, heads, seq, head_dim)


### 2. Generation WITHOUT a cache (the slow baseline)

**What this cell does (plain language):** greedy decoding where each step feeds the entire sequence so far back through the model and only uses the last position's logits. All the earlier tokens' keys and values are recomputed from scratch every single step - wasted work, since they cannot change (causal attention: past tokens never see the future).

In [2]:
@torch.no_grad()
def generate_nocache(model, ids, n_new):
    for _ in range(n_new):
        logits, _ = model(ids)                         # recompute EVERYTHING each step
        ids = torch.cat([ids, logits[:, -1].argmax(-1, keepdim=True)], dim=1)
    return ids

prompt = torch.randint(0, 64, (1, 16))
out_nc = generate_nocache(model, prompt, 20)
print("generated tokens:", out_nc[0, 16:].tolist())


generated tokens: [62, 42, 21, 9, 56, 52, 0, 19, 47, 28, 7, 19, 16, 36, 23, 27, 57, 54, 21, 9]


### 3. Generation WITH a KV cache

**What this cell does (plain language):** the same greedy loop, but now we run the prompt once (**prefill**) to build the cache, then feed only the single newest token each step (**decode**), reusing cached keys/values. We assert that it produces exactly the same tokens as the no-cache version and that the logits agree to float precision - a cache is an optimisation, never an approximation.

In [3]:
@torch.no_grad()
def generate_cache(model, ids, n_new, key_valid=None, pos=None):
    logits, cache = model(ids, key_valid=key_valid, pos=pos)       # prefill: process the whole prompt once
    out = [ids]
    last_pos = None if pos is None else pos[:, -1:]
    for _ in range(n_new):
        nxt = logits[:, -1].argmax(-1, keepdim=True)
        out.append(nxt)
        if key_valid is not None:
            key_valid = torch.cat([key_valid, torch.ones_like(nxt, dtype=torch.bool)], dim=1)
            last_pos = last_pos + 1
        logits, cache = model(nxt, cache=cache, key_valid=key_valid, pos=last_pos)   # decode: ONE token in
    return torch.cat(out, dim=1)

out_c = generate_cache(model, prompt, 20)
print("identical tokens with and without cache:", torch.equal(out_nc, out_c))

with torch.no_grad():                                              # logits equivalence for the last position
    full, _ = model(out_c[:, :30])
    _, c = model(out_c[:, :29]); step, _ = model(out_c[:, 29:30], cache=c)
print("max |logit diff| full-recompute vs cached step:", (full[:, -1] - step[:, -1]).abs().max().item())


identical tokens with and without cache: True
max |logit diff| full-recompute vs cached step: 3.337860107421875e-06


### 4. Timing: cache vs. no cache as the sequence grows

**What this cell does (plain language):** we time generating `n` new tokens both ways for increasing `n`. Without a cache step `t` processes `t` tokens, so total work grows quadratically; with a cache each step processes one token (attention still reads all `t` cached keys, but projections/MLPs are O(1) per step). Expect the gap to widen with `n`. Timings depend on the machine; the trend is what matters.

In [4]:
def timeit(fn, reps=2):
    fn()                                                         # warm-up
    best = float("inf")
    for _ in range(reps):
        t = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t)
    return best

print(f"{'new tokens':>10} {'no cache (s)':>13} {'cache (s)':>10} {'speedup':>8}")
for n in (16, 32, 64, 128, 192):
    t_nc = timeit(lambda: generate_nocache(model, prompt, n), reps=1)
    t_c = timeit(lambda: generate_cache(model, prompt, n), reps=1)
    print(f"{n:>10} {t_nc:>13.3f} {t_c:>10.3f} {t_nc / t_c:>8.1f}x")


new tokens  no cache (s)  cache (s)  speedup


        16         0.041      0.023      1.8x


        32         0.122      0.093      1.3x


        64         0.279      0.152      1.8x


       128         0.701      0.282      2.5x


       192         1.445      0.357      4.0x


### 5. KV cache memory: the formula and the real thing

**What this cell does (plain language):** the cache is not free: `2 * layers * heads * head_dim * seq_len * bytes` per sequence. We check the formula against the actual tensors, then evaluate it for a 7B-class configuration (32 layers, 32 heads, head_dim 128, fp16). At long contexts the cache of a single sequence is gigabytes - this is why serving systems spend so much effort on cache management (paged attention, grouped-query attention, quantized caches).

In [5]:
def kv_bytes(layers, heads, head_dim, seq, batch=1, bytes_per=4):
    return 2 * layers * heads * head_dim * seq * batch * bytes_per

_, cache = model(torch.randint(0, 64, (3, 50)))
actual = sum(k.numel() * k.element_size() + v.numel() * v.element_size() for k, v in cache)
print("formula:", kv_bytes(4, 4, 32, 50, batch=3), "bytes | measured:", actual, "bytes")

for seq in (2048, 4096, 32768):
    gb = kv_bytes(32, 32, 128, seq, bytes_per=2) / 1e9
    print(f"7B-class, fp16, context {seq:>6}: {gb:6.2f} GB per sequence  (weights themselves ~14 GB)")


formula: 614400 bytes | measured: 614400 bytes
7B-class, fp16, context   2048:   1.07 GB per sequence  (weights themselves ~14 GB)
7B-class, fp16, context   4096:   2.15 GB per sequence  (weights themselves ~14 GB)
7B-class, fp16, context  32768:  17.18 GB per sequence  (weights themselves ~14 GB)


### 6. Batching: throughput vs. batch size

**What this cell does (plain language):** decoding one sequence at a time leaves the hardware under-used because each step is dominated by overheads and by reading the weights. Running `B` sequences together reuses each weight load for `B` tokens. We measure tokens/second for several batch sizes with the same prompts length and the cache enabled. (On GPUs the effect is far larger than on this CPU; here we just see the trend.)

In [6]:
n_new = 48
print(f"{'batch':>6} {'time (s)':>9} {'tokens/s':>10}")
for B in (1, 2, 4, 8, 16, 32):
    p = torch.randint(0, 64, (B, 16))
    t = timeit(lambda: generate_cache(model, p, n_new), reps=1)
    print(f"{B:>6} {t:>9.3f} {B * n_new / t:>10.0f}")


 batch  time (s)   tokens/s
     1     0.064        749


     2     0.070       1378


     4     0.103       1870


     8     0.137       2793


    16     0.202       3802


    32     0.328       4678


### 7. Batching prompts of different lengths: left-padding, masks and position ids

**What this cell does (plain language):** real requests have different lengths. We **left-pad** shorter prompts so all end at the same column (the newest token must be last in every row), pass a `key_valid` mask so nothing attends to padding, and pass correct position ids (counting only real tokens). We then verify that each row of the padded batch generates exactly the same tokens as running that prompt alone. Try removing the mask or the positions to see it break - these are the most common batching bugs.

In [7]:
torch.manual_seed(1)
lens = [5, 9, 16, 12]
prompts = [torch.randint(1, 64, (1, n)) for n in lens]
Lmax = max(lens)
batch = torch.zeros(len(lens), Lmax, dtype=torch.long)           # 0 = pad id
valid = torch.zeros(len(lens), Lmax, dtype=torch.bool)
for i, p in enumerate(prompts):
    batch[i, Lmax - p.shape[1]:] = p[0]; valid[i, Lmax - p.shape[1]:] = True
pos = (valid.cumsum(1) - 1).clamp(min=0)                          # position ids ignore the left padding

out_batch = generate_cache(model, batch, 12, key_valid=valid, pos=pos)
ok = []
for i, p in enumerate(prompts):
    solo = generate_cache(model, p, 12)
    ok.append(torch.equal(solo[0, p.shape[1]:], out_batch[i, Lmax:]))
print("each padded-batch row matches its solo run:", ok)

wrong = generate_cache(model, batch, 12)                          # forgot mask + positions
print("WITHOUT mask/positions the short prompts diverge:",
      [torch.equal(generate_cache(model, p, 12)[0, p.shape[1]:], wrong[i, Lmax:]) for i, p in enumerate(prompts)])


each padded-batch row matches its solo run: [True, True, True, True]


WITHOUT mask/positions the short prompts diverge: [False, False, True, False]


### 8. Speculative decoding: the accept/reject rule keeps the target distribution exactly

**What this cell does (plain language):** the draft model proposes a token `x ~ q`; the target (distribution `p`) accepts it with probability `min(1, p(x)/q(x))`; on rejection a replacement is sampled from the *residual* distribution `norm(max(0, p - q))`. The surprising theorem: the resulting token is distributed exactly as `p`, whatever `q` is. We check it empirically on a 6-token vocabulary with 200,000 trials and also report the acceptance rate, which equals `sum_x min(p(x), q(x))`.

In [8]:
torch.manual_seed(0)
Vs = 6
p = torch.softmax(torch.randn(Vs) * 1.5, dim=0)                  # target distribution
q = torch.softmax(torch.randn(Vs) * 1.5, dim=0)                  # (different) draft distribution

N = 200000
x = torch.multinomial(q, N, replacement=True)                    # draft proposals
accept = torch.rand(N) < torch.minimum(torch.ones(N), p[x] / q[x])
residual = torch.clamp(p - q, min=0); residual = residual / residual.sum()
resampled = torch.multinomial(residual, N, replacement=True)
final = torch.where(accept, x, resampled)

emp = torch.bincount(final, minlength=Vs).float() / N
print("target p      :", p.numpy().round(4))
print("empirical out :", emp.numpy().round(4), "  (matches p)")
print("draft q       :", q.numpy().round(4), "  (what you'd get WITHOUT the correction)")
print("acceptance rate: empirical", round(accept.float().mean().item(), 4), "| theory sum(min(p,q)) =", round(torch.minimum(p, q).sum().item(), 4))


target p      : [0.7509 0.0479 0.0028 0.1746 0.0146 0.0091]
empirical out : [0.7529 0.0472 0.0027 0.1735 0.0147 0.009 ]   (matches p)
draft q       : [0.2302 0.4419 0.0427 0.0686 0.0514 0.1652]   (what you'd get WITHOUT the correction)
acceptance rate: empirical 0.3722 | theory sum(min(p,q)) = 0.3734


### 9. Greedy speculative decoding with real models: identical output, fewer target passes

**What this cell does (plain language):** now with actual models. The draft is a *perturbed copy* of the target (weights plus noise), a stand-in for a distilled small model. Each round: the draft greedily proposes `K` tokens; the target scores all of them in ONE forward pass; we keep the longest prefix where the target's own argmax agrees, then add the target's token at the first disagreement (or a bonus token if everything matched). For greedy decoding the output must equal plain target decoding token for token - we assert it - and we count how many tokens we get per target pass. (For clarity this version recomputes without a cache.)

In [9]:
def make_draft(target, noise):
    d = copy.deepcopy(target)
    with torch.no_grad():
        for p_ in d.parameters(): p_.add_(torch.randn_like(p_) * noise * p_.std())
    return d.eval()

@torch.no_grad()
def spec_greedy(target, draft, ids, n_new, K=4):
    start, passes, accepted_total, drafted_total = ids.shape[1], 0, 0, 0
    while ids.shape[1] - start < n_new:
        cand = ids
        for _ in range(K):                                             # draft proposes K tokens greedily
            cand = torch.cat([cand, draft(cand)[0][:, -1].argmax(-1, keepdim=True)], dim=1)
        tgt = target(cand)[0].argmax(-1)                               # ONE target pass scores all K positions
        passes += 1
        L = ids.shape[1]
        n_ok = 0
        while n_ok < K and cand[0, L + n_ok] == tgt[0, L + n_ok - 1]:   # draft token == target's own choice?
            n_ok += 1
        ids = torch.cat([cand[:, :L + n_ok], tgt[:, L + n_ok - 1: L + n_ok]], dim=1)   # + target's token (correction/bonus)
        accepted_total += n_ok; drafted_total += K
    return ids[:, :start + n_new], passes, accepted_total / drafted_total

prompt = torch.randint(0, 64, (1, 16)); n_new = 40
ref_out = generate_cache(model, prompt, n_new)
for noise in (0.0, 0.05, 0.2, 0.6):
    out, passes, rate = spec_greedy(model, make_draft(model, noise), prompt, n_new, K=4)
    print(f"draft noise {noise:<4} | same as target greedy: {torch.equal(out, ref_out)} | target passes {passes:>2} for {n_new} tokens "
          f"({n_new / passes:.2f} tok/pass) | per-token accept rate {rate:.2f}")


draft noise 0.0  | same as target greedy: True | target passes  8 for 40 tokens (5.00 tok/pass) | per-token accept rate 1.00


draft noise 0.05 | same as target greedy: True | target passes 11 for 40 tokens (3.64 tok/pass) | per-token accept rate 0.73


draft noise 0.2  | same as target greedy: True | target passes 15 for 40 tokens (2.67 tok/pass) | per-token accept rate 0.43


draft noise 0.6  | same as target greedy: True | target passes 30 for 40 tokens (1.33 tok/pass) | per-token accept rate 0.08


### 10. Predicting the speedup: tokens per pass and wall-clock model

**What this cell does (plain language):** accept rate `alpha`, draft length `K` and the draft/target cost ratio `c` determine the gain. Expected tokens per target pass is `(1 - alpha^(K+1)) / (1 - alpha)`; one round costs `1 + K*c` target-pass equivalents, so speedup = tokens-per-pass / (1 + K*c). We tabulate it for `c = 0.1` and see the trade-off: more drafted tokens help only if acceptance is high - otherwise you pay for drafts that get thrown away. There is an optimal `K`.

In [10]:
def expected_speedup(alpha, K, c):
    tokens_per_pass = (1 - alpha ** (K + 1)) / (1 - alpha) if alpha < 1 else K + 1
    return tokens_per_pass / (1 + K * c)

c = 0.1
print("speedup (draft cost = 10% of target)     K=1    K=2    K=4    K=8   K=16")
for alpha in (0.3, 0.5, 0.7, 0.9, 0.95):
    print(f"  accept rate alpha = {alpha:<4}            " + "  ".join(f"{expected_speedup(alpha, K, c):5.2f}" for K in (1, 2, 4, 8, 16)))
best = max(range(1, 33), key=lambda K: expected_speedup(0.7, K, c))
print("best K for alpha=0.7, c=0.1:", best, "->", round(expected_speedup(0.7, best, c), 2), "x")


speedup (draft cost = 10% of target)     K=1    K=2    K=4    K=8   K=16
  accept rate alpha = 0.3              1.18   1.16   1.02   0.79   0.55
  accept rate alpha = 0.5              1.36   1.46   1.38   1.11   0.77
  accept rate alpha = 0.7              1.55   1.82   1.98   1.78   1.28
  accept rate alpha = 0.9              1.73   2.26   2.93   3.40   3.20
  accept rate alpha = 0.95             1.77   2.38   3.23   4.11   4.48
best K for alpha=0.7, c=0.1: 4 -> 1.98 x


## Common bugs

- **Forgetting to feed only the new token when using the cache**
  Passing the whole sequence plus the cache re-appends old keys/values, duplicating them and corrupting attention. Fix: after prefill, feed exactly the newest token(s) together with the cache.

- **Wrong position ids with cache**
  With a cache the new token's position is `past_len`, not 0. Forgetting this makes every decoded token look like position 0 and the output degrades silently. Fix: derive positions from the cache length (as `GPT.forward` does) or pass them explicitly.

- **Right-padding for batched generation**
  With right padding the 'last token' of short prompts is a pad, so you generate after padding. Fix: left-pad, mask padding keys, and compute position ids from the mask.

- **Not masking padding keys**
  Pad tokens then participate in attention and change outputs of real tokens (section 7 shows the divergence). Fix: pass a key mask, and use a large negative number (not `-inf`) if rows can be fully masked.

- **Cache not reset between requests**
  Reusing a cache from a previous prompt leaks context across users/prompts. Fix: create a fresh cache per request (or track per-sequence lengths in a paged cache).

- **Timing without warm-up or sync**
  The first call includes allocation/kernel setup; on GPU, kernels are asynchronous so `time.perf_counter()` alone measures launch not execution. Fix: warm up, and call `torch.cuda.synchronize()` before reading the clock on GPU.

- **Speculative decoding with a wrong correction rule**
  Using `max(0, p - q)` without normalising, or simply taking the target's argmax on rejection in *sampling* mode, changes the output distribution. Fix: sample from `norm(max(0, p - q))` as verified in section 8.

- **Measuring tokens/sec including the prefill**
  Prefill is compute-bound and decode is memory-bound; mixing them hides both. Fix: report time-to-first-token and per-token decode latency separately.

## Exercises

Attempt each one in the empty cell right after the question. My full solution is in the cell after that - don't peek until you've tried.

**Exercise 1.** Measure the time of the prefill step and of a single cached decode step separately for prompt lengths 16, 128 and 512. Which one grows with the prompt length?

In [11]:
# Your attempt for Exercise 1 here


**Solution 1**

In [12]:
# Solution 1
for L in (16, 128, 512):
    pr_ = torch.randint(0, 64, (1, L))
    t_pre = timeit(lambda: model(pr_), reps=2)
    _, c_ = model(pr_)
    nxt = torch.randint(0, 64, (1, 1))
    t_dec = timeit(lambda: model(nxt, cache=c_), reps=3)
    print(f"prompt {L:>3}: prefill {t_pre*1000:7.2f} ms | one decode step {t_dec*1000:6.2f} ms")


prompt  16: prefill    2.55 ms | one decode step   1.68 ms
prompt 128: prefill    7.11 ms | one decode step   2.02 ms


prompt 512: prefill   64.34 ms | one decode step   3.03 ms


**Exercise 2.** Compute how many sequences of context 4096 fit in 24 GB for the 7B-class fp16 config of section 5 if the weights take 14 GB. Then redo it with grouped-query attention (8 KV heads instead of 32).

In [13]:
# Your attempt for Exercise 2 here


**Solution 2**

In [14]:
# Solution 2
free = 24e9 - 14e9
mha = kv_bytes(32, 32, 128, 4096, bytes_per=2)
gqa = kv_bytes(32, 8, 128, 4096, bytes_per=2)
print("MHA sequences:", int(free // mha), "| GQA(8 kv heads) sequences:", int(free // gqa))


MHA sequences: 4 | GQA(8 kv heads) sequences: 18


**Exercise 3.** Implement a *sliding-window* cache that keeps only the last W=32 positions per layer, and show that the cache stays at a constant size during generation (output will differ from full attention - why?).

In [15]:
# Your attempt for Exercise 3 here


**Solution 3**

In [16]:
# Solution 3
@torch.no_grad()
def generate_window(model, ids, n_new, W=32):
    logits, cache = model(ids)
    out = [ids]; sizes = []
    for _ in range(n_new):
        nxt = logits[:, -1].argmax(-1, keepdim=True); out.append(nxt)
        pos = torch.tensor([[ids.shape[1] + len(out) - 2]])        # absolute position of the new token
        logits, cache = model(nxt, cache=cache, pos=pos)
        cache = [(k[:, :, -W:], v[:, :, -W:]) for k, v in cache]   # evict old entries
        sizes.append(cache[0][0].shape[2])
    return torch.cat(out, dim=1), sizes
_, sizes = generate_window(model, torch.randint(0, 64, (1, 16)), 60)
print("cache length over time (first 5, last 3):", sizes[:5], sizes[-3:])
# Outputs differ because tokens older than W are no longer visible to attention.


cache length over time (first 5, last 3): [17, 18, 19, 20, 21] [32, 32, 32]


**Exercise 4.** Verify the batching equivalence for 3 prompts when you add a *fourth* very short prompt of length 1. Does the property still hold?

In [17]:
# Your attempt for Exercise 4 here


**Solution 4**

In [18]:
# Solution 4
prompts2 = prompts + [torch.randint(1, 64, (1, 1))]
Lm = max(pi.shape[1] for pi in prompts2)
b2 = torch.zeros(len(prompts2), Lm, dtype=torch.long); v2 = torch.zeros(len(prompts2), Lm, dtype=torch.bool)
for i, pi in enumerate(prompts2): b2[i, Lm - pi.shape[1]:] = pi[0]; v2[i, Lm - pi.shape[1]:] = True
o2 = generate_cache(model, b2, 10, key_valid=v2, pos=(v2.cumsum(1) - 1).clamp(min=0))
print([torch.equal(generate_cache(model, pi, 10)[0, pi.shape[1]:], o2[i, Lm:]) for i, pi in enumerate(prompts2)])


[True, True, True, True, True]


**Exercise 5.** Run the empirical test of section 8 with `q = p` (perfect draft) and with a `q` that is nearly the opposite of `p`. Report the acceptance rate in each case and confirm the output still matches `p`.

In [19]:
# Your attempt for Exercise 5 here


**Solution 5**

In [20]:
# Solution 5
for name, qq in [("q = p", p.clone()), ("q ~ reversed p", torch.softmax(-p.log() * 1.0, dim=0))]:
    x = torch.multinomial(qq, N, replacement=True)
    acc = torch.rand(N) < torch.clamp(p[x] / qq[x], max=1)
    res = torch.clamp(p - qq, min=0)
    res = res / res.sum() if res.sum() > 0 else p
    fin = torch.where(acc, x, torch.multinomial(res, N, replacement=True))
    e = torch.bincount(fin, minlength=Vs).float() / N
    print(f"{name:<15} accept {acc.float().mean().item():.3f} | max |emp - p| = {(e - p).abs().max().item():.4f}")


q = p           accept 1.000 | max |emp - p| = 0.0008
q ~ reversed p  accept 0.075 | max |emp - p| = 0.0006


**Exercise 6.** Using `expected_speedup`, find the minimum acceptance rate `alpha` for which speculative decoding (K=4) beats plain decoding when the draft costs 30% of the target (c=0.3).

In [21]:
# Your attempt for Exercise 6 here


**Solution 6**

In [22]:
# Solution 6
for alpha in [i / 100 for i in range(5, 100, 5)]:
    if expected_speedup(alpha, 4, 0.3) > 1:
        print("break-even alpha ~", alpha, "-> speedup", round(expected_speedup(alpha, 4, 0.3), 2)); break


break-even alpha ~ 0.6 -> speedup 1.05
